In [3]:
# Step 1: Install necessary libraries
!pip install transformers[torch] datasets evaluate rouge_score sentencepiece accelerate tqdm sacrebleu


  Preparing metadata (setup.py) ... done
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 84.1/84.1 kB 3.3 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 193.6/193.6 kB 4.4 MB/s eta 0:00:00a 0:00:01
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 363.4/363.4 MB 4.4 MB/s eta 0:00:00:00:0100:01
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 13.8/13.8 MB 68.1 MB/s eta 0:00:00:00:0100:01
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 24.6/24.6 MB 47.8 MB/s eta 0:00:0000:0100:01m
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 883.7/883.7 kB 33.5 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 664.8/664.8 MB 2.1 MB/s eta 0:00:00:00:0100:01
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 211.5/211.5 MB 7.2 MB/s eta 0:00:00:00:0100:01
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 56.3/56.3 MB 3.5 MB/s eta 0:00:000:00:0100:01
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 127.9/127.9 MB 13.0 MB/s eta 0:00:0000:0100:01
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 207.5/207.5 MB 7.7 MB/s eta 0:00:

In [4]:
pip install sacrebleu

     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 51.8/51.8 kB 2.8 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 104.1/104.1 kB 5.6 MB/s eta 0:00:00
Note: you may need to restart the kernel to use updated packages.


In [5]:
import os
import zipfile
from tqdm.auto import tqdm
import torch
import pandas as pd
from datasets import load_dataset, Dataset
from transformers import (
    AutoTokenizer,
    AutoModelForSeq2SeqLM,
    AutoModelForCausalLM,
    DataCollatorForSeq2Seq,
    DataCollatorForLanguageModeling,
    Seq2SeqTrainingArguments,
    TrainingArguments,
    Seq2SeqTrainer,
    Trainer,
    EarlyStoppingCallback,
    PrinterCallback,
)
import evaluate
from datetime import datetime
import time
import numpy as np


2025-07-24 21:08:23.159433: E external/local_xla/xla/stream_executor/cuda/cuda_fft.cc:477] Unable to register cuFFT factory: Attempting to register factory for plugin cuFFT when one has already been registered
E0000 00:00:1753391303.345865      36 cuda_dnn.cc:8310] Unable to register cuDNN factory: Attempting to register factory for plugin cuDNN when one has already been registered
E0000 00:00:1753391303.402612      36 cuda_blas.cc:1418] Unable to register cuBLAS factory: Attempting to register factory for plugin cuBLAS when one has already been registered


In [ ]:
from transformers import TrainerCallback

class PrinterCallback(TrainerCallback):
    def on_epoch_begin(self, args, state, control, **kwargs):
        print(f"\n--- Starting Epoch {int(state.epoch)} ---")

    def on_evaluate(self, args, state, control, metrics, **kwargs):
        print(f"\nEvaluation results for epoch {int(state.epoch)}: {metrics}")
# Silence warnings and set local cache paths
os.environ["TRANSFORMERS_NO_ADVISORY_WARNINGS"] = "true"
os.environ["HF_DATASETS_CACHE"] = "./hf_cache"
os.environ["TRANSFORMERS_CACHE"] = "./transformers_cache"

# Show device info
print("--- Environment Information ---")
print("CUDA available:", torch.cuda.is_available())
if torch.cuda.is_available():
    print("Using GPU:", torch.cuda.get_device_name(0))
else:
    print("Using CPU only")
print("-----------------------------")



In [44]:
import time
import torch
import evaluate
import numpy as np
from datetime import datetime
from datasets import load_dataset
from transformers import (
    AutoTokenizer, 
    AutoModelForSeq2SeqLM,
    AutoModelForCausalLM,
    Seq2SeqTrainingArguments, 
    Seq2SeqTrainer,          
    DataCollatorForSeq2Seq,
    EarlyStoppingCallback,PrinterCallback
)


# --- Helper Functions & Metrics ---
def clean_summarization_dataset(examples):
    code = examples.get("whole_func_string")
    docstring = examples.get("func_documentation_string")
    return code is not None and len(code.strip()) > 0 and docstring is not None and len(docstring.strip()) > 0

def preprocess_seq2seq(examples, tokenizer):
    prefix = "summarize python: "
    inputs = [prefix + code for code in examples["whole_func_string"]]
    targets = [docstring for docstring in examples["func_documentation_string"]]
    model_inputs = tokenizer(inputs, text_target=targets, max_length=512, truncation=True)
    return model_inputs

def preprocess_causal_lm_for_generation(examples, tokenizer):
    """Prepares data for Causal LM generation by creating a prompt."""
    inputs = [f"Generate a summary for this Python code: {code}{tokenizer.eos_token}" for code in examples["whole_func_string"]]
    targets = [docstring for docstring in examples["func_documentation_string"]]
    model_inputs = tokenizer(inputs, text_target=targets, max_length=512, truncation=True)
    return model_inputs

rouge = evaluate.load("rouge")
bleu = evaluate.load("sacrebleu")
def compute_generation_metrics(eval_pred, tokenizer):
    predictions, labels = eval_pred
    labels = np.where(labels != -100, labels, tokenizer.pad_token_id)
    decoded_preds = tokenizer.batch_decode(predictions, skip_special_tokens=True)
    decoded_labels = tokenizer.batch_decode(labels, skip_special_tokens=True)
    rouge_result = rouge.compute(predictions=decoded_preds, references=decoded_labels, use_stemmer=True)
    bleu_references = [[label] for label in decoded_labels]
    bleu_result = bleu.compute(predictions=decoded_preds, references=bleu_references)
    result = { "rouge1": rouge_result["rouge1"], "rouge2": rouge_result["rouge2"], "rougeL": rouge_result["rougeL"], "bleu": bleu_result["score"] }
    return result

# --- Data Loading ---
print("Loading and preparing summarization dataset...")
raw_train_dataset = load_dataset("code_search_net", "python", split='train')
cleaned_dataset = raw_train_dataset.filter(clean_summarization_dataset)
split_dataset = cleaned_dataset.train_test_split(test_size=0.1, seed=42)
train_dataset_full = split_dataset['train']
eval_dataset_full = split_dataset['test']

# --- Model Dictionaries ---
SEQ2SEQ_MODELS = { "codet5-small": "Salesforce/codet5-small" }
CAUSAL_LM_MODELS = { "codeparrot-small": "codeparrot/codeparrot-small" }


# --- 1. Fine-tuning for Seq2Seq Models ---
print("\n\n--- Starting Fine-tuning for Seq2Seq Models ---")
for model_name, model_id in SEQ2SEQ_MODELS.items():
    print(f"\n--- Training Model: {model_name} ({model_id}) ---")
    tokenizer = AutoTokenizer.from_pretrained(model_id)
    model = AutoModelForSeq2SeqLM.from_pretrained(model_id)
    
    small_train_dataset = train_dataset_full.shuffle(seed=42).select(range(20000))
    small_eval_dataset = eval_dataset_full.shuffle(seed=42).select(range(2000))

    train_dataset = small_train_dataset.map(lambda x: preprocess_seq2seq(x, tokenizer), batched=True, remove_columns=small_train_dataset.column_names)
    eval_dataset = small_eval_dataset.map(lambda x: preprocess_seq2seq(x, tokenizer), batched=True, remove_columns=small_eval_dataset.column_names)

    data_collator = DataCollatorForSeq2Seq(tokenizer=tokenizer, model=model)
    
    training_args = Seq2SeqTrainingArguments(
        output_dir=f"./outputs_{model_name}_summarization",
        per_device_train_batch_size=8, per_device_eval_batch_size=8,
        predict_with_generate=True, learning_rate=2e-5, num_train_epochs=15,
        save_strategy="epoch", eval_strategy="epoch", logging_strategy="epoch",
        load_best_model_at_end=True, metric_for_best_model="rougeL", greater_is_better=True,
        report_to="none"
    )

    trainer = Seq2SeqTrainer(
        model=model, args=training_args,
        train_dataset=train_dataset, eval_dataset=eval_dataset,
        tokenizer=tokenizer, data_collator=data_collator,
        compute_metrics=lambda p: compute_generation_metrics(p, tokenizer),
        callbacks=[PrinterCallback(), EarlyStoppingCallback(early_stopping_patience=2)],
    )

    trainer.train()


# --- 2. Fine-tuning for Causal LM Models ---
print("\n\n--- Starting Fine-tuning for Causal LM Models ---")
for model_name, model_id in CAUSAL_LM_MODELS.items():
    print(f"\n--- Training Model: {model_name} ({model_id}) ---")
    
    tokenizer = AutoTokenizer.from_pretrained(model_id)
    if tokenizer.pad_token is None:
        tokenizer.pad_token = tokenizer.eos_token
    model = AutoModelForCausalLM.from_pretrained(model_id)
    
    small_train_dataset = train_dataset_full.shuffle(seed=42).select(range(20000))
    small_eval_dataset = eval_dataset_full.shuffle(seed=42).select(range(2000))
    
    train_dataset = small_train_dataset.map(lambda x: preprocess_causal_lm_for_generation(x, tokenizer), batched=True, remove_columns=small_train_dataset.column_names)
    eval_dataset = small_eval_dataset.map(lambda x: preprocess_causal_lm_for_generation(x, tokenizer), batched=True, remove_columns=small_eval_dataset.column_names)

    data_collator = DataCollatorForSeq2Seq(tokenizer=tokenizer, model=model)
    
    training_args = Seq2SeqTrainingArguments(
        output_dir=f"./outputs_{model_name}_summarization",
        per_device_train_batch_size=2, per_device_eval_batch_size=2, 
        gradient_accumulation_steps=8,
        predict_with_generate=True, 
        learning_rate=2e-5, num_train_epochs=5,
        save_strategy="epoch", eval_strategy="epoch", logging_strategy="epoch",
        load_best_model_at_end=True, metric_for_best_model="rougeL", greater_is_better=True,
        report_to="none"
    )

    trainer = Seq2SeqTrainer(
        model=model, args=training_args,
        train_dataset=train_dataset, eval_dataset=eval_dataset,
        tokenizer=tokenizer, data_collator=data_collator,
        compute_metrics=lambda p: compute_generation_metrics(p, tokenizer), 
        callbacks=[PrinterCallback(), EarlyStoppingCallback(early_stopping_patience=2)],
    )

    trainer.train()

Loading and preparing summarization dataset...


--- Starting Fine-tuning for Seq2Seq Models ---

--- Training Model: codet5-small (Salesforce/codet5-small) ---


/tmp/ipykernel_36/2732291126.py:88: FutureWarning: `tokenizer` is deprecated and will be removed in version 5.0.0 for `Seq2SeqTrainer.__init__`. Use `processing_class` instead.
  trainer = Seq2SeqTrainer(


Epoch,Training Loss,Validation Loss,Rouge1,Rouge2,Rougel,Bleu
1,0.192100,0.041470,0.679832,0.665568,0.679956,3.826028
2,0.056300,0.033943,0.680108,0.665884,0.680210,3.826228
3,0.044400,0.032023,0.680171,0.665992,0.680263,3.831440
4,0.037300,0.031358,0.680560,0.666515,0.680664,3.830389
5,0.033500,0.029926,0.680560,0.666516,0.680668,3.832029


{'loss': 0.1921, 'grad_norm': 1.67951238155365, 'learning_rate': 1.866986666666667e-05, 'epoch': 1.0}
{'eval_loss': 0.041469987481832504, 'eval_rouge1': 0.6798322843354979, 'eval_rouge2': 0.6655675406885867, 'eval_rougeL': 0.6799555970300171, 'eval_bleu': 3.826028234431503, 'eval_runtime': 96.7729, 'eval_samples_per_second': 20.667, 'eval_steps_per_second': 2.583, 'epoch': 1.0}
{'loss': 0.0563, 'grad_norm': 0.1866559088230133, 'learning_rate': 1.7336533333333333e-05, 'epoch': 2.0}
{'eval_loss': 0.03394334018230438, 'eval_rouge1': 0.6801077937033868, 'eval_rouge2': 0.6658836661174344, 'eval_rougeL': 0.6802098994122743, 'eval_bleu': 3.826228434168487, 'eval_runtime': 97.1317, 'eval_samples_per_second': 20.591, 'eval_steps_per_second': 2.574, 'epoch': 2.0}
{'loss': 0.0444, 'grad_norm': 2.0585105419158936, 'learning_rate': 1.60032e-05, 'epoch': 3.0}
{'eval_loss': 0.03202338516712189, 'eval_rouge1': 0.6801713117466586, 'eval_rouge2': 0.6659924461645219, 'eval_rougeL': 0.6802629793057949, 'e

KeyboardInterrupt: 

In [45]:
last_checkpoint = "/kaggle/working/outputs_codet5-small_summarization/checkpoint-12500"

# Resume training from that checkpoint
trainer.train(resume_from_checkpoint=last_checkpoint)

There were missing keys in the checkpoint model loaded: ['encoder.embed_tokens.weight', 'decoder.embed_tokens.weight', 'lm_head.weight'].


Epoch,Training Loss,Validation Loss,Rouge1,Rouge2,Rougel,Bleu
6,0.030400,0.028847,0.680734,0.666714,0.680837,3.833195
7,0.028400,0.029615,0.680760,0.666715,0.680850,3.833372
8,0.026500,0.028509,0.680725,0.666687,0.680837,3.833034
9,0.025400,0.028034,0.680725,0.666687,0.680837,3.833034


{'loss': 0.0304, 'grad_norm': 0.050921566784381866, 'learning_rate': 1.2003733333333333e-05, 'epoch': 6.0}
{'eval_loss': 0.028847068548202515, 'eval_rouge1': 0.6807344171007153, 'eval_rouge2': 0.6667135558702558, 'eval_rougeL': 0.6808371216985957, 'eval_bleu': 3.8331951790239023, 'eval_runtime': 97.2454, 'eval_samples_per_second': 20.567, 'eval_steps_per_second': 2.571, 'epoch': 6.0}
{'loss': 0.0284, 'grad_norm': 0.029751872643828392, 'learning_rate': 1.0670933333333333e-05, 'epoch': 7.0}
{'eval_loss': 0.02961496263742447, 'eval_rouge1': 0.6807600581263564, 'eval_rouge2': 0.6667147196767337, 'eval_rougeL': 0.6808499422114163, 'eval_bleu': 3.833371588470914, 'eval_runtime': 96.8407, 'eval_samples_per_second': 20.652, 'eval_steps_per_second': 2.582, 'epoch': 7.0}
{'loss': 0.0265, 'grad_norm': 0.5412884950637817, 'learning_rate': 9.338666666666667e-06, 'epoch': 8.0}
{'eval_loss': 0.02850918658077717, 'eval_rouge1': 0.6807246185199337, 'eval_rouge2': 0.6666865572781306, 'eval_rougeL': 0.68

There were missing keys in the checkpoint model loaded: ['encoder.embed_tokens.weight', 'decoder.embed_tokens.weight', 'lm_head.weight'].


{'train_runtime': 2986.0644, 'train_samples_per_second': 100.467, 'train_steps_per_second': 12.558, 'train_loss': 0.012294890509711371, 'epoch': 9.0}


TrainOutput(global_step=22500, training_loss=0.012294890509711371, metrics={'train_runtime': 2986.0644, 'train_samples_per_second': 100.467, 'train_steps_per_second': 12.558, 'train_loss': 0.012294890509711371, 'epoch': 9.0})

In [16]:
import time
import torch
import evaluate
import numpy as np
from datetime import datetime
from datasets import load_dataset
from transformers import (
    AutoTokenizer, 
    AutoModelForSeq2SeqLM,
    AutoModelForCausalLM,
    Seq2SeqTrainingArguments, 
    Seq2SeqTrainer,          
    DataCollatorForSeq2Seq,
    EarlyStoppingCallback,PrinterCallback
)


# --- Helper Functions & Metrics ---
def clean_summarization_dataset(examples):
    code = examples.get("whole_func_string")
    docstring = examples.get("func_documentation_string")
    return code is not None and len(code.strip()) > 0 and docstring is not None and len(docstring.strip()) > 0

def preprocess_seq2seq(examples, tokenizer):
    prefix = "summarize python: "
    inputs = [prefix + code for code in examples["whole_func_string"]]
    targets = [docstring for docstring in examples["func_documentation_string"]]
    model_inputs = tokenizer(inputs, text_target=targets, max_length=512, truncation=True,padding=True)
    return model_inputs

def preprocess_causal_lm_for_generation(examples, tokenizer):
    """Prepares data for Causal LM generation by creating a prompt."""
    inputs = [f"Generate a summary for this Python code: {code}{tokenizer.eos_token}" for code in examples["whole_func_string"]]
    targets = [docstring for docstring in examples["func_documentation_string"]]
    model_inputs = tokenizer(inputs, text_target=targets, max_length=512, truncation=True,padding=True)
    return model_inputs

rouge = evaluate.load("rouge")
bleu = evaluate.load("sacrebleu")
def compute_generation_metrics(eval_pred, tokenizer):
    predictions, labels = eval_pred
    labels = np.where(labels != -100, labels, tokenizer.pad_token_id)
    decoded_preds = tokenizer.batch_decode(predictions, skip_special_tokens=True)
    decoded_labels = tokenizer.batch_decode(labels, skip_special_tokens=True)
    rouge_result = rouge.compute(predictions=decoded_preds, references=decoded_labels, use_stemmer=True)
    bleu_references = [[label] for label in decoded_labels]
    bleu_result = bleu.compute(predictions=decoded_preds, references=bleu_references)
    result = { "rouge1": rouge_result["rouge1"], "rouge2": rouge_result["rouge2"], "rougeL": rouge_result["rougeL"], "bleu": bleu_result["score"] }
    return result

# --- Data Loading ---
print("Loading and preparing summarization dataset...")
raw_train_dataset = load_dataset("code_search_net", "python", split='train')
cleaned_dataset = raw_train_dataset.filter(clean_summarization_dataset)
split_dataset = cleaned_dataset.train_test_split(test_size=0.1, seed=42)
train_dataset_full = split_dataset['train']
eval_dataset_full = split_dataset['test']

Loading and preparing summarization dataset...


In [19]:
CAUSAL_LM_MODELS = { "codeparrot-small": "codeparrot/codeparrot-small", "tinycodelm": "upiter/TinyCodeLM-150M" }

# --- 2. Fine-tuning for Causal LM Models ---
print("\n\n--- Starting Fine-tuning for Causal LM Models ---")
for model_name, model_id in CAUSAL_LM_MODELS.items():
    print(f"\n--- Training Model: {model_name} ({model_id}) ---")
    
    tokenizer = AutoTokenizer.from_pretrained(model_id,padding_side='left)
    if tokenizer.pad_token is None:
        tokenizer.pad_token = tokenizer.eos_token
    model = AutoModelForCausalLM.from_pretrained(model_id)
    
    small_train_dataset = train_dataset_full.shuffle(seed=42).select(range(20000))
    small_eval_dataset = eval_dataset_full.shuffle(seed=42).select(range(2000))
    
    train_dataset = small_train_dataset.map(lambda x: preprocess_causal_lm_for_generation(x, tokenizer), batched=True, remove_columns=small_train_dataset.column_names)
    eval_dataset = small_eval_dataset.map(lambda x: preprocess_causal_lm_for_generation(x, tokenizer), batched=True, remove_columns=small_eval_dataset.column_names)

    data_collator = DataCollatorForSeq2Seq(tokenizer=tokenizer, model=model)
    
    training_args = Seq2SeqTrainingArguments(
        output_dir=f"./outputs_{model_name}_summarization",
        per_device_train_batch_size=2, per_device_eval_batch_size=2, 
        gradient_accumulation_steps=8,
        predict_with_generate=True, 
        learning_rate=2e-5, num_train_epochs=10,
        save_strategy="epoch", eval_strategy="epoch", logging_strategy="epoch",
        load_best_model_at_end=True, metric_for_best_model="rougeL", greater_is_better=True,
        report_to="none"
    )

    trainer = Seq2SeqTrainer(
        model=model, args=training_args,
        train_dataset=train_dataset, eval_dataset=eval_dataset,
        tokenizer=tokenizer, data_collator=data_collator,
        compute_metrics=lambda p: compute_generation_metrics(p, tokenizer), 
        callbacks=[PrinterCallback(), EarlyStoppingCallback(early_stopping_patience=1)],
    )

    trainer.train()
    trainer.save_model(f"./{model_name}_summarization_model")
    tokenizer.save_pretrained(f"./{model_name}_summarization_model")
   



--- Starting Fine-tuning for Causal LM Models ---

--- Training Model: codeparrot-small (codeparrot/codeparrot-small) ---


Map:   0%|          | 0/1000 [00:00<?, ? examples/s]

/tmp/ipykernel_36/3135840224.py:32: FutureWarning: `tokenizer` is deprecated and will be removed in version 5.0.0 for `Seq2SeqTrainer.__init__`. Use `processing_class` instead.
  trainer = Seq2SeqTrainer(


Epoch,Training Loss,Validation Loss,Rouge1,Rouge2,Rougel,Bleu
1,0.992900,0.899813,0.435383,0.421524,0.435116,26.939952
2,0.880700,0.887379,0.435383,0.421524,0.435116,26.942640


Setting `pad_token_id` to `eos_token_id`:50256 for open-end generation.


...

{'loss': 0.9178, 'grad_norm': 0.6272713541984558, 'learning_rate': 1.60032e-05, 'epoch': 2.0}


A decoder-only architecture is being used, but right-padding was detected! For correct generation results, please set `padding_side='left'` when initializing the tokenizer.
A decoder-only architecture is being used, but right-padding was detected! For correct generation results, please set `padding_side='left'` when initializing the tokenizer.
A decoder-only architecture is being used, but right-padding was detected! For correct generation results, please set `padding_side='left'` when initializing the tokenizer.
A decoder-only architecture is being used, but right-padding was detected! For correct generation results, please set `padding_side='left'` when initializing the tokenizer.
A decoder-only architecture is being used, but right-padding was detected! For correct generation results, please set `padding_side='left'` when initializing the tokenizer.
A decoder-only architecture is being used, but right-padding was detected! For correct generation results, please set `padding_side='le

{'eval_loss': 0.9335935115814209, 'eval_rouge1': 0.437766151866969, 'eval_rouge2': 0.4240011072555452, 'eval_rougeL': 0.437720384386228, 'eval_bleu': 27.34890775230986, 'eval_runtime': 161.182, 'eval_samples_per_second': 6.204, 'eval_steps_per_second': 3.102, 'epoch': 2.0}


There were missing keys in the checkpoint model loaded: ['lm_head.weight'].


{'train_runtime': 2310.5522, 'train_samples_per_second': 43.28, 'train_steps_per_second': 2.705, 'train_loss': 1.06933193359375, 'epoch': 2.0}


In [13]:
import time
import torch
import evaluate
import numpy as np
from datetime import datetime
from datasets import load_dataset
from transformers import (
    AutoTokenizer,
    AutoModelForSeq2SeqLM,
    AutoModelForCausalLM,
    TrainingArguments,
    Trainer,
    DataCollatorForLanguageModeling, # Changed from DataCollatorForSeq2Seq
    DataCollatorForSeq2Seq,
    EarlyStoppingCallback
)


# --- Helper Functions & Metrics ---
def clean_summarization_dataset(examples):
    code = examples.get("whole_func_string")
    docstring = examples.get("func_documentation_string")
    return code is not None and len(code.strip()) > 0 and docstring is not None and len(docstring.strip()) > 0

# This function is now specific to Causal LM fine-tuning
def preprocess_causal_lm_finetuning(examples, tokenizer):
    """
    Prepares data for Causal LM fine-tuning by concatenating prompt and completion.
    The loss is masked for the prompt part.
    """
    # Format the inputs and targets
    inputs = [f"Generate a summary for this Python code: {code}\n\n{docstring}{tokenizer.eos_token}"
              for code, docstring in zip(examples["whole_func_string"], examples["func_documentation_string"])]
    
    # Tokenize the combined text. This handles padding and truncation for the batch.
    model_inputs = tokenizer(inputs, max_length=512, truncation=True, padding=True)

    # Create labels by cloning the input_ids
    labels = model_inputs["input_ids"].copy()
    model_inputs["labels"] = labels

    # We need to mask the prompt part for each example in the batch.
    # To do this, we'll tokenize the prompts separately to get their lengths.
    prompts = [f"Generate a summary for this Python code: {code}\n\n"
               for code in examples["whole_func_string"]]
    
    # Tokenize prompts without padding to find their actual lengths.
    prompt_token_lengths = [len(tokenizer(p, add_special_tokens=False)['input_ids']) for p in prompts]

    # Mask the prompt tokens in the labels tensor.
    for i in range(len(model_inputs["labels"])):
        prompt_len = prompt_token_lengths[i]
        # Set the prompt part of the labels to -100
        model_inputs["labels"][i][:prompt_len] = [-100] * prompt_len
            
    return model_inputs

rouge = evaluate.load("rouge")
bleu = evaluate.load("sacrebleu")
def compute_generation_metrics(eval_pred, tokenizer):
    predictions, labels = eval_pred
    # In case -100 is still in predictions, replace it with pad_token_id
    predictions = np.where(predictions != -100, predictions, tokenizer.pad_token_id)
    labels = np.where(labels != -100, labels, tokenizer.pad_token_id)
    
    decoded_preds = tokenizer.batch_decode(predictions, skip_special_tokens=True)
    decoded_labels = tokenizer.batch_decode(labels, skip_special_tokens=True)
    
    # ROUGE expects a list of string predictions and references
    rouge_result = rouge.compute(predictions=decoded_preds, references=decoded_labels, use_stemmer=True)
    
    # BLEU expects a list of string predictions and a list of lists of string references
    bleu_references = [[label] for label in decoded_labels]
    bleu_result = bleu.compute(predictions=decoded_preds, references=bleu_references)
    
    result = { "rouge1": rouge_result["rouge1"], "rouge2": rouge_result["rouge2"], "rougeL": rouge_result["rougeL"], "bleu": bleu_result["score"] }
    return result

# --- Data Loading ---
print("Loading and preparing summarization dataset...")
raw_train_dataset = load_dataset("code_search_net", "python", split='train')
cleaned_dataset = raw_train_dataset.filter(clean_summarization_dataset)
split_dataset = cleaned_dataset.train_test_split(test_size=0.1, seed=42)
train_dataset_full = split_dataset['train']
eval_dataset_full = split_dataset['test']
CAUSAL_LM_MODELS = { "codeparrot-small": "codeparrot/codeparrot-small" }


# --- Fine-tuning for Causal LM Models ---
print("\n\n--- Starting Fine-tuning for Causal LM Models ---")
for model_name, model_id in CAUSAL_LM_MODELS.items():
    print(f"\n--- Training Model: {model_name} ({model_id}) ---")
    
    tokenizer = AutoTokenizer.from_pretrained(model_id)
    if tokenizer.pad_token is None:
        tokenizer.pad_token = tokenizer.eos_token
    model = AutoModelForCausalLM.from_pretrained(model_id)
    # For demonstration, using a smaller subset. Adjust as needed.
    small_train_dataset = train_dataset_full.shuffle(seed=42).select(range(20000))
    small_eval_dataset = eval_dataset_full.shuffle(seed=42).select(range(2000))
    
    # **FIX 1: Use the new preprocessing function**
    train_dataset = small_train_dataset.map(lambda x: preprocess_causal_lm_finetuning(x, tokenizer), batched=True, remove_columns=small_train_dataset.column_names)
    eval_dataset = small_eval_dataset.map(lambda x: preprocess_causal_lm_finetuning(x, tokenizer), batched=True, remove_columns=small_eval_dataset.column_names)

    # **FIX 2: Use the correct data collator for Causal LM**
    data_collator = DataCollatorForLanguageModeling(tokenizer=tokenizer, mlm=False)
    
    training_args = TrainingArguments(
        output_dir=f"./outputs_{model_name}_summarization",
        per_device_train_batch_size=2, 
        per_device_eval_batch_size=2,
        gradient_accumulation_steps=8,
        learning_rate=2e-5, 
        num_train_epochs=5, # Reduced epochs for faster demonstration
        save_strategy="epoch", 
        eval_strategy="epoch", 
        logging_strategy="epoch",
        load_best_model_at_end=True, 
        metric_for_best_model="rougeL", 
        greater_is_better=True,
        report_to="none"
    )

    trainer = Trainer(
        model=model, 
        args=training_args,
        train_dataset=train_dataset, 
        eval_dataset=eval_dataset,
        tokenizer=tokenizer, 
        data_collator=data_collator,
        compute_metrics=lambda p: compute_generation_metrics(p, tokenizer),
        callbacks=[EarlyStoppingCallback(early_stopping_patience=2)],
    )

    trainer.train()
    trainer.save_model(f"./{model_name}_summarization_model")
    tokenizer.save_pretrained(f"./{model_name}_summarization_model")
   

Loading and preparing summarization dataset...


--- Starting Fine-tuning for Causal LM Models ---

--- Training Model: codeparrot-small (codeparrot/codeparrot-small) ---


Map:   0%|          | 0/20000 [00:00<?, ? examples/s]

Token indices sequence length is longer than the specified maximum sequence length for this model (1031 > 1024). Running this sequence through the model will result in indexing errors


Map:   0%|          | 0/2000 [00:00<?, ? examples/s]

/tmp/ipykernel_36/4255287084.py:125: FutureWarning: `tokenizer` is deprecated and will be removed in version 5.0.0 for `Trainer.__init__`. Use `processing_class` instead.
  trainer = Trainer(


ValueError: Unable to create tensor, you should probably activate truncation and/or padding with 'padding=True' 'truncation=True' to have batched tensors with the same length. Perhaps your features (`attention_mask` in this case) have excessive nesting (inputs type `list` where type `int` is expected).

In [ ]:


# --- Helper Functions & Metrics ---
def clean_code_generation_dataset(examples):
    """Checks if the examples from the code generation dataset are valid."""
    intent = examples.get("intent")
    snippet = examples.get("snippet")
    return intent is not None and len(intent.strip()) > 0 and snippet is not None and len(snippet.strip()) > 0

def preprocess_seq2seq_codegen(examples, tokenizer):
    """Prepares data for Seq2Seq models for the code generation task."""
    prefix = "Generate Python code: "
    inputs = [prefix + intent for intent in examples["intent"]]
    targets = [snippet for snippet in examples["snippet"]]
    model_inputs = tokenizer(inputs, text_target=targets, max_length=512, truncation=True)
    return model_inputs

def preprocess_causal_lm_codegen(examples, tokenizer):
    """Prepares data for Causal LM for the code generation task by creating a prompt."""
    inputs = [f"Generate Python code for this intent: {intent}{tokenizer.eos_token}" for intent in examples["intent"]]
    targets = [snippet for snippet in examples["snippet"]]
    model_inputs = tokenizer(inputs, text_target=targets, max_length=512, truncation=True)
    return model_inputs

# Load metrics
rouge = evaluate.load("rouge")
bleu = evaluate.load("sacrebleu")
exact_match = evaluate.load("exact_match")

def compute_generation_metrics(eval_pred, tokenizer):
    """Computes ROUGE, BLEU, and Exact Match for generated code."""
    predictions, labels = eval_pred
    # In case the model returns more than the prediction, we slice it
    if isinstance(predictions, tuple):
        predictions = predictions[0]

    labels = np.where(labels != -100, labels, tokenizer.pad_token_id)
    decoded_preds = tokenizer.batch_decode(predictions, skip_special_tokens=True)
    decoded_labels = tokenizer.batch_decode(labels, skip_special_tokens=True)

    # ROUGE
    rouge_result = rouge.compute(predictions=decoded_preds, references=decoded_labels, use_stemmer=True)

    # BLEU
    bleu_references = [[label] for label in decoded_labels]
    bleu_result = bleu.compute(predictions=decoded_preds, references=bleu_references)

    # Exact Match
    em_result = exact_match.compute(predictions=decoded_preds, references=decoded_labels)

    result = {
        "rouge1": rouge_result["rouge1"],
        "rouge2": rouge_result["rouge2"],
        "rougeL": rouge_result["rougeL"],
        "bleu": bleu_result["score"],
        "exact_match": em_result["exact_match"]
    }
    return result

# --- Data Loading ---
print("Loading and preparing code generation dataset...")
# Using the Conala (Code/Natural Language) dataset, curated version
train_dataset_full = load_dataset("neulab/conala", "curated", split='train')
eval_dataset_full = load_dataset("neulab/conala", "curated", split='test')

# Clean the datasets
train_dataset_full = train_dataset_full.filter(clean_code_generation_dataset)
eval_dataset_full = eval_dataset_full.filter(clean_code_generation_dataset)


# --- Model Dictionaries ---
SEQ2SEQ_MODELS = { "codet5-small": "Salesforce/codet5-small" }
CAUSAL_LM_MODELS = { "codeparrot-small": "codeparrot/codeparrot-small", "tinycodelm": "upiter/TinyCodeLM-150M" }


# --- 1. Fine-tuning for Seq2Seq Models ---
print("\n\n--- Starting Fine-tuning for Seq2Seq Models ---")
for model_name, model_id in SEQ2SEQ_MODELS.items():
    print(f"\n--- Training Model: {model_name} ({model_id}) ---")
    tokenizer = AutoTokenizer.from_pretrained(model_id)
    model = AutoModelForSeq2SeqLM.from_pretrained(model_id)

    # Using smaller subsets for faster training demonstration
    small_train_dataset = train_dataset_full.shuffle(seed=42).select(range(1000))
    small_eval_dataset = eval_dataset_full.shuffle(seed=42).select(range(200))

    train_dataset = small_train_dataset.map(lambda x: preprocess_seq2seq_codegen(x, tokenizer), batched=True, remove_columns=small_train_dataset.column_names)
    eval_dataset = small_eval_dataset.map(lambda x: preprocess_seq2seq_codegen(x, tokenizer), batched=True, remove_columns=small_eval_dataset.column_names)

    data_collator = DataCollatorForSeq2Seq(tokenizer=tokenizer, model=model)

    training_args = Seq2SeqTrainingArguments(
        output_dir=f"./outputs_{model_name}_generation",
        per_device_train_batch_size=8, per_device_eval_batch_size=8,
        predict_with_generate=True, learning_rate=2e-5, num_train_epochs=15,
        generation_max_length=128, # Set max length for generated code
        save_strategy="epoch", eval_strategy="epoch", logging_strategy="epoch",
        load_best_model_at_end=True, metric_for_best_model="exact_match", greater_is_better=True,
        report_to="none"
    )

    trainer = Seq2SeqTrainer(
        model=model, args=training_args,
        train_dataset=train_dataset, eval_dataset=eval_dataset,
        tokenizer=tokenizer, data_collator=data_collator,
        compute_metrics=lambda p: compute_generation_metrics(p, tokenizer),
        callbacks=[PrinterCallback(), EarlyStoppingCallback(early_stopping_patience=2)],
    )

    trainer.train()


# --- 2. Fine-tuning for Causal LM Models ---
print("\n\n--- Starting Fine-tuning for Causal LM Models ---")
for model_name, model_id in CAUSAL_LM_MODELS.items():
    print(f"\n--- Training Model: {model_name} ({model_id}) ---")

    tokenizer = AutoTokenizer.from_pretrained(model_id)
    if tokenizer.pad_token is None:
        tokenizer.pad_token = tokenizer.eos_token
    model = AutoModelForCausalLM.from_pretrained(model_id)

    # Using smaller subsets for faster training demonstration
    small_train_dataset = train_dataset_full.shuffle(seed=42).select(range(1000))
    small_eval_dataset = eval_dataset_full.shuffle(seed=42).select(range(200))

    train_dataset = small_train_dataset.map(lambda x: preprocess_causal_lm_codegen(x, tokenizer), batched=True, remove_columns=small_train_dataset.column_names)
    eval_dataset = small_eval_dataset.map(lambda x: preprocess_causal_lm_codegen(x, tokenizer), batched=True, remove_columns=small_eval_dataset.column_names)

    # Using DataCollatorForSeq2Seq is fine for Causal LMs in this setup
    data_collator = DataCollatorForSeq2Seq(tokenizer=tokenizer, model=model, padding="longest")

    training_args = Seq2SeqTrainingArguments(
        output_dir=f"./outputs_{model_name}_generation",
        per_device_train_batch_size=2, per_device_eval_batch_size=2, # Causal LMs are larger, may need smaller batch size
        gradient_accumulation_steps=8,
        predict_with_generate=True,
        learning_rate=2e-5, num_train_epochs=10,
        generation_max_length=128, # Set max length for generated code
        save_strategy="epoch", eval_strategy="epoch", logging_strategy="epoch",
        load_best_model_at_end=True, metric_for_best_model="exact_match", greater_is_better=True,
        report_to="none"
    )

    trainer = Seq2SeqTrainer(
        model=model, args=training_args,
        train_dataset=train_dataset, eval_dataset=eval_dataset,
        tokenizer=tokenizer, data_collator=data_collator,
        compute_metrics=lambda p: compute_generation_metrics(p, tokenizer),
        callbacks=[PrinterCallback(), EarlyStoppingCallback(early_stopping_patience=2)],
    )

    trainer.train()